# H1 proxy: class-sampling fine-tune from the locked RAMP epoch-80 weights

Per seed, 10 more epochs of RAMP (same flags as `ramp_scratch`, lr 0.005 = the lr the weights ended at, momentum restarts):
control `ramp_cont` on GPU 0 versus `ramp_samp` (= control plus `--aat_sample`, budget-neutral class sampling) on GPU 1.
Both arms are evaluated with the unchanged fixed-eps AutoAttack protocol and compared with each other and with the epoch-80 baseline.
Needs the `HF_TOKEN` secret, GPU T4 x2, Internet on. Expected about 2.5 session-hours for the seed pair; low power for per-class effects.

In [ ]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
os.environ['TIME_BUDGET_H'] = '10.5'             # training stops cleanly after this and pushes its checkpoint
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('install', 'bash scripts/ramp_official.sh install')   # dependencies + full import check, before anything else
stage('data', 'bash scripts/ramp_official.sh data')         # CIFAR-10 (cached on Hugging Face after the first run)

def finished(run):
    return os.path.exists(f'external/ramp/trained_models/{run}/log_eval_final.txt')

stage('ramp_samp_train', 'bash scripts/ramp_official.sh trainpair \\"0 1\\"')
for s in (0, 1):  # sampling weights trajectory
    f = f'external/ramp/trained_models/rampsamp_s{s}/log_train.txt'
    if os.path.exists(f):
        print(f'--- seed {s}'); print(''.join([l for l in open(f) if l.startswith('[aat-samp]')][-3:]))
if all(finished(f'{r}_s{s}') for r in ('rampcont', 'rampsamp') for s in (0, 1)):
    stage('ramp_samp_eval', 'bash scripts/ramp_official.sh evalpair \\"0 1\\" && '
          'python scripts/compare_runs.py --names SAMP CONT '
          '--a \\"runs_official/rampsamp_s*/eval_autoattack.json\\" --b \\"runs_official/rampcont_s*/eval_autoattack.json\\" && '
          'python scripts/compare_runs.py --names SAMP RAMP80 '
          '--a \\"runs_official/rampsamp_s*/eval_autoattack.json\\" '
          '--b \\"baselines/results/ramp_scratch_l5_s*_eval_autoattack.json\\"')
else:
    print('Training not finished; check the logs.')

In [ ]:
## Control versus class sampler, seeds 0 and 1